# 00 — Context v3: Frequency Multiplexing Enables Quantum Scale

**Version marker:** `00_context_v3_architecture`

**Seminar:** Integrated Microcombs for Quantum Applications  
**Speaker:** Xu Yi, University of Virginia

This notebook frames the repository question:

> **Which resource scales quantum systems: more devices or more modes?**

Notebook 00 is architectural. It does not model full quantum optics, fabrication, loss, detection, or Kerr dynamics.

The central claim is:

\[
\text{frequency multiplexing changes the scaling resource from devices to modes}
\]

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

VERSION = "00_context_v3_architecture"
print("running:", VERSION)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
else:
    ROOT = cwd

FIGURES_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
CSV_DIR = RESULTS_DIR / "csv"
JSON_DIR = RESULTS_DIR / "json"

for path in [FIGURES_DIR, CSV_DIR, JSON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("FIGURES_DIR:", FIGURES_DIR)

## 1. Architecture table

The question is not whether both approaches can label \(N\) channels.

Both can.

The question is which resource has to scale:

- replicated physical devices
- or frequency-indexed modes

In [ ]:
architecture_table = pd.DataFrame([
    {
        "resource": "Pump / source",
        "scale_by_devices": "N sources",
        "scale_by_modes": "1 pump",
        "interpretation": "mode scaling reuses the source path"
    },
    {
        "resource": "Physical devices",
        "scale_by_devices": "N devices",
        "scale_by_modes": "1 integrated resonator",
        "interpretation": "mode scaling increases mode count inside one device"
    },
    {
        "resource": "Frequency modes",
        "scale_by_devices": "not primary scaling resource",
        "scale_by_modes": "N frequency-indexed modes",
        "interpretation": "frequency becomes the multiplexing dimension"
    },
    {
        "resource": "Quantum channels",
        "scale_by_devices": "N channels",
        "scale_by_modes": "N channels",
        "interpretation": "same channel target; different scaling resource"
    },
    {
        "resource": "Applications",
        "scale_by_devices": "replicated optical systems",
        "scale_by_modes": "computing, networking, sensing from mode multiplexing",
        "interpretation": "architecture shifts from hardware replication to mode addressing"
    },
])

architecture_table

In [ ]:
architecture_table_path = CSV_DIR / "00_v3_architecture_table.csv"
architecture_table.to_csv(architecture_table_path, index=False)
print("saved:", architecture_table_path)

## 2. Architecture comparison

This is the notebook's hero figure.

It shows the device-replication architecture against the microcomb mode-scaling architecture.

In [ ]:
def draw_box(ax, xy, text, width=0.34, height=0.13, fontsize=11):
    x, y = xy
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.025,rounding_size=0.025",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)

def draw_arrow(ax, x, y_top, y_bottom):
    ax.annotate("", xy=(x, y_bottom), xytext=(x, y_top), arrowprops=dict(arrowstyle="->", linewidth=1.8))

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

ax.text(0.25, 0.94, "Scale by Devices", ha="center", va="center", fontsize=16, fontweight="bold")
ax.text(0.75, 0.94, "Scale by Modes", ha="center", va="center", fontsize=16, fontweight="bold")

left_x, right_x = 0.08, 0.58
y_positions = [0.76, 0.56, 0.36, 0.16]

left_labels = ["N channels", "N sources", "N detectors", "N optical paths"]
right_labels = ["1 pump", "1 resonator", "N frequency modes", "N quantum channels"]

for y, label in zip(y_positions, left_labels):
    draw_box(ax, (left_x, y), label)

for y, label in zip(y_positions, right_labels):
    draw_box(ax, (right_x, y), label)

for ys in zip(y_positions[:-1], y_positions[1:]):
    draw_arrow(ax, left_x + 0.17, ys[0], ys[1] + 0.13)
    draw_arrow(ax, right_x + 0.17, ys[0], ys[1] + 0.13)

ax.text(0.5, 0.50, "vs", ha="center", va="center", fontsize=18, fontweight="bold")
ax.text(
    0.5,
    0.04,
    "Frequency multiplexing changes the scaling resource: devices → modes.",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
)

fig.tight_layout()

architecture_path = FIGURES_DIR / "00_v3_architecture_comparison.png"
fig.savefig(architecture_path, dpi=200)
plt.show()

print("saved:", architecture_path)

## 3. Channel density per physical device

Line plots of \(N\) versus \(1\) are mathematically correct, but not very informative.

A more salient architecture quantity is:

\[
\text{channels per physical device}
\]

In a replicated device architecture, the simplified channel density is approximately constant.

In a mode-scaling architecture, the channel density grows with addressable frequency modes.

In [ ]:
devices = np.array([1, 10, 100])
classical_channels = devices
classical_channels_per_device = classical_channels / devices

microcomb_devices = np.ones_like(devices)
microcomb_modes = devices
microcomb_channels = microcomb_modes
microcomb_channels_per_device = microcomb_channels / microcomb_devices

density_table = pd.DataFrame({
    "scale_reference": devices,
    "classical_devices": devices,
    "classical_channels": classical_channels,
    "classical_channels_per_device": classical_channels_per_device,
    "microcomb_devices": microcomb_devices,
    "microcomb_frequency_modes": microcomb_modes,
    "microcomb_channels": microcomb_channels,
    "microcomb_channels_per_device": microcomb_channels_per_device,
})

density_table

In [ ]:
density_path = CSV_DIR / "00_v3_channel_density.csv"
density_table.to_csv(density_path, index=False)
print("saved:", density_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(devices))
width = 0.35

ax.bar(x - width/2, classical_channels_per_device, width, label="Scale by devices")
ax.bar(x + width/2, microcomb_channels_per_device, width, label="Scale by modes")

ax.set_title("Channel Density per Physical Device")
ax.set_xlabel("Reference channel scale")
ax.set_ylabel("Channels per physical device")
ax.set_xticks(x)
ax.set_xticklabels([str(v) for v in devices])
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig.tight_layout()

density_figure_path = FIGURES_DIR / "00_v3_channel_density.png"
fig.savefig(density_figure_path, dpi=200)
plt.show()

print("saved:", density_figure_path)

## 4. Multiplexing ladder

This figure mirrors the seminar logic:

\[
\text{pump}
\rightarrow
\text{microresonator}
\rightarrow
\text{frequency comb}
\rightarrow
\text{many modes}
\rightarrow
\text{many quantum channels}
\rightarrow
\text{applications}
\]

In [ ]:
ladder = [
    "Pump",
    "Microresonator",
    "Frequency comb",
    "N frequency modes",
    "N quantum channels",
    "Computing · Networking · Sensing",
]

fig, ax = plt.subplots(figsize=(11, 3))
ax.set_xlim(0, len(ladder))
ax.set_ylim(0, 1)
ax.axis("off")

for i, label in enumerate(ladder):
    x = i + 0.1
    box = FancyBboxPatch(
        (x, 0.35),
        0.8,
        0.3,
        boxstyle="round,pad=0.03,rounding_size=0.03",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)
    ax.text(x + 0.4, 0.5, label, ha="center", va="center", fontsize=10, wrap=True)

    if i < len(ladder) - 1:
        ax.annotate(
            "",
            xy=(i + 1.05, 0.5),
            xytext=(i + 0.92, 0.5),
            arrowprops=dict(arrowstyle="->", linewidth=1.8),
        )

ax.set_title("Multiplexing Ladder", fontsize=14, fontweight="bold")

fig.tight_layout()

ladder_path = FIGURES_DIR / "00_v3_multiplexing_ladder.png"
fig.savefig(ladder_path, dpi=200)
plt.show()

print("saved:", ladder_path)

## 5. Frequency comb architecture

A microcomb supplies a ladder of frequency modes:

\[
f_n = f_0 + n\Delta f
\]

Notebook 00 shows the comb architecture only.

Kerr-pair labels such as \((-1,+1)\), \((-2,+2)\), and \((-3,+3)\) belong in Notebook 13.

In [ ]:
mode_indices = np.arange(-10, 11)

mode_labels = []
for n in mode_indices:
    if n == 0:
        mode_labels.append("f₀")
    elif n < 0:
        mode_labels.append(f"f₀{n}Δf")
    else:
        mode_labels.append(f"f₀+{n}Δf")

mode_table = pd.DataFrame({
    "mode_index_n": mode_indices,
    "relative_frequency": mode_indices,
    "frequency_label": mode_labels,
    "role": np.where(mode_indices == 0, "pump", "frequency mode")
})

mode_table_path = CSV_DIR / "00_v3_frequency_comb_modes.csv"
mode_table.to_csv(mode_table_path, index=False)

mode_table.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

heights = np.ones_like(mode_indices, dtype=float)
ax.vlines(mode_indices, 0, heights, linewidth=1)
ax.scatter(mode_indices, heights, s=28)

ax.axvline(0, linestyle="--", alpha=0.6)
ax.text(0, 1.12, "pump\nf₀", ha="center", va="bottom")

for idx, n in enumerate(range(1, 6)):
    y = 0.76 - idx * 0.08
    ax.plot([-n, n], [y, y], linewidth=1.2)
    ax.scatter([-n, n], [y, y], s=10)

tick_positions = [-10, -5, -1, 0, 1, 5, 10]
tick_labels = ["f₀−10Δf", "f₀−5Δf", "f₀−Δf", "f₀", "f₀+Δf", "f₀+5Δf", "f₀+10Δf"]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

ax.set_title("Frequency Comb Architecture")
ax.set_xlabel("Frequency mode")
ax.set_yticks([])
ax.set_ylim(0, 1.25)
ax.set_xlim(mode_indices.min() - 1, mode_indices.max() + 1)

fig.tight_layout()

comb_path = FIGURES_DIR / "00_v3_frequency_comb.png"
fig.savefig(comb_path, dpi=200)
plt.show()

print("saved:", comb_path)

## 6. Summary

Classical scaling increases physical device count.

Microcomb scaling increases frequency-mode count.

This repository explores whether frequency multiplexing changes the scaling architecture of quantum systems.

In [ ]:
summary = {
    "notebook": "00_context_v3_architecture",
    "version": VERSION,
    "title": "Frequency Multiplexing Enables Quantum Scale",
    "seminar": "Integrated Microcombs for Quantum Applications",
    "speaker": "Xu Yi, University of Virginia",
    "repo_question": "Which resource scales quantum systems: more devices or more modes?",
    "architectural_claim": "Frequency multiplexing changes the scaling resource from replicated devices to frequency-indexed modes.",
    "outputs": [
        "figures/00_v3_architecture_comparison.png",
        "figures/00_v3_channel_density.png",
        "figures/00_v3_multiplexing_ladder.png",
        "figures/00_v3_frequency_comb.png",
        "results/csv/00_v3_architecture_table.csv",
        "results/csv/00_v3_channel_density.csv",
        "results/csv/00_v3_frequency_comb_modes.csv",
        "results/json/00_v3_context_summary.json"
    ]
}

summary_path = JSON_DIR / "00_v3_context_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))

In [ ]:
outputs = [
    FIGURES_DIR / "00_v3_architecture_comparison.png",
    FIGURES_DIR / "00_v3_channel_density.png",
    FIGURES_DIR / "00_v3_multiplexing_ladder.png",
    FIGURES_DIR / "00_v3_frequency_comb.png",
    CSV_DIR / "00_v3_architecture_table.csv",
    CSV_DIR / "00_v3_channel_density.csv",
    CSV_DIR / "00_v3_frequency_comb_modes.csv",
    JSON_DIR / "00_v3_context_summary.json",
]

for output in outputs:
    print("exists:", output.exists(), "→", output.relative_to(ROOT) if output.exists() else output)

## Takeaway

The notebook's central result is architectural:

\[
\text{Scale by devices:}\quad N\ \text{channels} \rightarrow N\ \text{source paths}
\]

\[
\text{Scale by modes:}\quad 1\ \text{resonator} \rightarrow N\ \text{frequency modes} \rightarrow N\ \text{channels}
\]

This motivates the next notebooks:

- **07:** frequency comb architecture
- **13:** Kerr pair generation
- **23:** multipartite entanglement networks
- **37:** scaling by modes vs devices